# Map Template System Demo

This notebook demonstrates the improved template system using enums and auto-registration.

In [ ]:
from AoE2ScenarioParser.datasets.players import PlayerId
from AoE2ScenarioParser.datasets.units import UnitInfo
from AoE2ScenarioParser.datasets.buildings import BuildingInfo
from AoE2ScenarioParser.datasets.terrains import TerrainId
from aoe2mapgenerator.common.enums.enum import MapLayerType, GateType
from aoe2mapgenerator.map.map_manager import MapManager
from aoe2mapgenerator.templates.template_types import TemplateType
from aoe2mapgenerator.common.constants.constants import BASE_SCENE_DIR_WINDOWS_WSL
import os

## Create Map Manager

Create a new map manager with a size of 256x256

In [ ]:
map_size = 256
map_manager = MapManager(map_size)

## Using Templates with Enum Values

Now we can use templates with enum values, which provides better type safety and IDE auto-completion

In [ ]:
# Create voronoi zones for our map
zones = map_manager.place_voronoi_zones({
    'number_of_regions': 12
}).get_dictionary(MapLayerType.ZONE)

In [ ]:
# Get the zone IDs
zone_ids = list(zones.keys())

## Apply Different Templates to Different Zones

Using our new method chaining approach

In [ ]:
import random

# Create forts in some zones
for i, zone_id in enumerate(zone_ids[:3]):
    # Create point collection for this zone
    points = map_manager.select_points({
        'map_layer_type': MapLayerType.ZONE,
        'map_object': zone_id,
        'number_of_points': 1,  # Get central point
        'area_radius': 10,
    })
    
    if points:
        center_point = points[0]
        # Use enum for template type
        map_manager.apply_template(
            TemplateType.FORT,
            center_point=center_point,
            size=30,
            player_id=PlayerId(i+1),
            gate_type=GateType.FORTIFIED_GATE
        )

In [ ]:
# Create forests in other zones
for i, zone_id in enumerate(zone_ids[3:6]):
    # Create point collection for this zone
    points = map_manager.select_points({
        'map_layer_type': MapLayerType.ZONE,
        'map_object': zone_id,
        'number_of_points': 1,  # Get central point
        'area_radius': 10,
    })
    
    if points:
        center_point = points[0]
        # Use the convenience method for forest
        map_manager.create_forest(
            center_point=center_point,
            size=40,
            forest_type=TemplateType.OAK_FOREST,
            player_id=PlayerId.GAIA
        )

In [ ]:
# Create villages in remaining zones
for i, zone_id in enumerate(zone_ids[6:9]):
    # Create point collection for this zone
    points = map_manager.select_points({
        'map_layer_type': MapLayerType.ZONE,
        'map_object': zone_id,
        'number_of_points': 1,  # Get central point
        'area_radius': 10,
    })
    
    if points:
        center_point = points[0]
        # Use the convenience method for villages
        map_manager.create_village(
            center_point=center_point,
            size=25,
            player_id=PlayerId(i+4)  # Different players
        )

## Method Chaining Example

Our new design makes it easy to chain template applications and other map operations

In [ ]:
# Chain multiple operations together
map_manager.create_fort(center_point=(50, 50), player_id=PlayerId.ONE) \
    .create_forest(center_point=(100, 100), forest_type=TemplateType.SNOW_FOREST) \
    .create_village(center_point=(150, 150), player_id=PlayerId.THREE) \
    .create_walls(center_point=(200, 200), player_id=PlayerId.FIVE)

## Write the Map to a File

Save our map with the applied templates

In [ ]:
output_file = os.path.join(BASE_SCENE_DIR_WINDOWS_WSL, "template_demo.aoe2scenario")
map_manager.write_map_and_save(output_file)
print(f"Map saved to {output_file}")

## Template System Benefits

The new template system offers several advantages:

1. **Type Safety**: Using enums prevents typos in template names
2. **Auto-completion**: IDEs can suggest available templates through enum values
3. **Method Chaining**: All operations return the map manager for cleaner code
4. **Auto-registration**: Templates register themselves with the decorator
5. **Separation of Concerns**: Templates are defined separately from the manager
6. **Convenience Methods**: Common templates have dedicated methods
7. **Extensibility**: New templates can be added easily without modifying existing code